In [1]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math

init_printing(use_latex=True)

In [2]:
import re
from IPython.display import display, Math

def color_terms(latex_str):
    latex_str = re.sub(
        r'(\\phi_\{REF\}\{\\left\(.+?\\right\)\})',
        r'{\\color{purple} \1}',
        latex_str
    )
    # Color all q_i(t) terms orange
    latex_str = re.sub(
        r'(q_\{([1])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{red} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([2])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([3])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{yellow} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([A])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Cyan} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([B])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Aquamarine} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([C])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{SpringGreen} \1{\\left(\3\\right)}}',
        latex_str
    )
    return latex_str

In [3]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

dtau12, dtau21, dtau13, dtau31, dtau23, dtau32 = symbols(
    r'\dot\tau_{12} \dot\tau_{21} \dot\tau_{13} \dot\tau_{31} \dot\tau_{23} \dot\tau_{32}',
    real=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

For now, we assume frequencies to be constant in time. Hartwig et. al. did a bit of analysis how this affects the clock noise removal.

In [4]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')
phiREF  = Function(r'\phi_{REF}')

epsilonA = Function(r'\epsilon_A')  # clock noise, s/c 1 (master)
epsilonB = Function(r'\epsilon_B')  # clock noise, s/c 2
epsilonC = Function(r'\epsilon_C')  # clock noise, s/c 3

omegaREFA, omegaREFB, omegaREFC = symbols(r'\omega^{REF}_A \omega^{REF}_B \omega^{REF}_C', real=True)

q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')

N1 = Function('N_{1}')
N2 = Function('N_{2}')
N3 = Function('N_{3}')

N1_SB = Function('N_{1}^{SB}')
N2_SB = Function('N_{2}^{SB}')
N3_SB = Function('N_{3}^{SB}')

In [5]:
tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

dtau = {
    (1,2): dtau12, (2,1): dtau21,
    (1,3): dtau13, (3,1): dtau31,
    (2,3): dtau23, (3,2): dtau32,
}


def D(expr, tau_key):
    """
    Apply a single delay for link tau_key.
    - If tau_key is a tuple (i,j): uses time-varying delay tau_ij(t) = tau_ij^0 + dtau_ij * t
    - If tau_key is a sympy symbol: applies it as a constant delay
    """
    if isinstance(tau_key, tuple):
        tau_val = tau[tau_key] + dtau[tau_key] * t
    else:
        tau_val = tau_key
    return expr.subs(t, t - tau_val)


def Dn(expr, *tau_keys):
    """
    Apply a sequence of delays in order, each evaluated at the
    already-shifted time from previous delays.
    
    Usage:
        Dn(expr, (1,2), (2,3))        # D_12 then D_23, time-varying
        Dn(expr, (1,2), tau23)        # D_12 time-varying, tau23 constant
        Dn(expr, tau12, tau23)        # both constant
    
    The delays are applied left-to-right, so Dn(f, (1,2), (2,3)) gives
    f(t - tau_12(t) - tau_23(t - tau_12(t))), which for linearly
    changing delays expands as:
        f(t - tau_12^0 - dtau_12*t - tau_23^0 - dtau_23*(t - tau_12^0 - dtau_12*t))
    """
    for tau_key in tau_keys:
        expr = D(expr, tau_key)
    return expr

In [6]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, omegaREFA, N1, N1_SB),
    2: (phi2, q2, omega2, omega2m, epsilonB, omegaREFB, N2, N2_SB),
    3: (phi3, q3, omega3, omega3m, epsilonC, omegaREFC, N3, N3_SB),
}


eta    = {}
etaSB  = {}
etaU  = {} # unmixed
REF  = {}
r = {}

include_phi = True 
include_ref_laser = False
include_clock_noise = False
include_board_jitter = False 
include_optical_noise = True  

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i, omegaREFi, N_i, N_i_SB = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, omegaREFj, N_j, N_j_SB = sc[j]

    phi_terms = (D(phi_j(t) - int(include_ref_laser) * phiREF(t), (i,j)) - (phi_i(t) - int(include_ref_laser) * phiREF(t))) if include_phi else 0

    clock_terms_C = (-(om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (-(om_j - om_i + omm_j - omm_i)*q_i(t)
                      - omm_i*q_i(t)
                      + omm_j * D(q_j(t), (i,j))) if include_clock_noise else 0

    board_terms_c  = (om_j * (eps_i(t) - D(eps_i(t), (i,j)))) if include_board_jitter else 0
    board_terms_SB = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), (i,j)))) if include_board_jitter else 0

    optical_terms = (N_i(t) - D(N_j(t), (i,j))) if include_optical_noise else 0 
    optical_terms_SB = (N_i_SB(t) - D(N_j_SB(t), (i,j))) if include_optical_noise else 0       

    eta[(i,j)] = (phi_terms + clock_terms_C + board_terms_c + optical_terms)

    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + board_terms_SB + optical_terms_SB), [q_i(t), q_j(t)])

    r[(i,j)] = simplify((eta[(i,j)] - etaSB[(i,j)]) / omm_j)

    #etaU[(i,j)] = (D(phi_j(t), t_ij) - om_j  * q_i(t) + (om_j * (eps_i(t) - D(eps_i(t), t_ij))))
    
    REF[(i,j)] = ( - int(include_clock_noise)*q_j(t) + int(include_board_jitter) * eps_i(t) ) # REF measured by the other PM, missing omegaREFi  *

The $\epsilon(t)$ terms originate from clock noise on the delay line boards which we will use three of. So we have A, B and C. Each corresponds to one spacecraft so the board A is where we apply delays 2 $\rightarrow$ 1 and 3 $\rightarrow$ 1. The $q(t)$ terms come from the Mokus. These are LISA-like, as in LISA will see similat clock noise coupling from the PM. All clock terms are given w.r.t the global clock, which we won't really ever know.

In [7]:
if True:
    for (i, j) in tau:
        display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + color_terms(latex(eta[(i,j)]))))
        #display(Math(r'\eta_{' + str(i) + str(j) + '}^{U} = ' + color_terms(latex(etaU[(i,j)]))))
        display(Math(r'\eta_{' + str(i) + str(j) + '}^{SB} = ' + color_terms(latex(etaSB[(i,j)]))))
        display(Math(r'\frac{r_{' + str(i) + str(j) + r'}}{\omega_{' + str(j) + '}^m} = ' + color_terms(latex(r[(i,j)]))))
        display(Math(r'\frac{REF_{' + str(i) + str(j) + r'}}{ \omega_{' + str(j) + '}^m} = ' + color_terms(latex(REF[(i,j)]))))
        print("------------------------------------------------------")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


In [8]:
# D_131     : 3->1 first, then 1->3  → Dn(expr, (3,1),(1,3))
# D_13121   : 2->1, 1->2, 3->1, 1->3 → Dn(expr, (2,1),(1,2),(3,1),(1,3))
# D_1213131 : 3->1, 1->3, 3->1, 1->3, 2->1, 1->2 → Dn(expr, (3,1),(1,3),(3,1),(1,3),(2,1),(1,2))

# D_121     : 2->1, 1->2             → Dn(expr, (2,1),(1,2))
# D_12131   : 3->1, 1->3, 2->1, 1->2 → Dn(expr, (3,1),(1,3),(2,1),(1,2))
# D_1312121 : 2->1, 1->2, 2->1, 1->2, 3->1, 1->3 → Dn(expr, (2,1),(1,2),(2,1),(1,2),(3,1),(1,3))

def P12_X2(expr):
    return -(  expr
             - Dn(expr, (3,1),(1,3))
             - Dn(expr, (2,1),(1,2),(3,1),(1,3))
             + Dn(expr, (3,1),(1,3),(3,1),(1,3),(2,1),(1,2)))

def P13_X2(expr):
    return  (  expr
             - Dn(expr, (2,1),(1,2))
             - Dn(expr, (3,1),(1,3),(2,1),(1,2))
             + Dn(expr, (2,1),(1,2),(2,1),(1,2),(3,1),(1,3)))

def P21_X2(expr): return P12_X2(Dn(expr, (1,2)))
def P31_X2(expr): return P13_X2(Dn(expr, (1,3)))
def P23_X2(expr): return 0
def P32_X2(expr): return 0

X2 = collect(expand(
      P12_X2(eta[(1,2)]) + P21_X2(eta[(2,1)])
    + P13_X2(eta[(1,3)]) + P31_X2(eta[(3,1)])
), [q1(t), q2(t), q3(t)])

In [9]:
from sympy import preorder_traversal, Function
from sympy.core.function import AppliedUndef
def drop_dtau_squared_in_args(expr):
    dtau_set = {dtau12, dtau21, dtau13, dtau31}
    
    def clean_arg(arg):
        result = 0
        for term in expand(arg).as_ordered_terms():
            # Count total degree in all dtau variables combined
            dtau_degree = sum(term.as_powers_dict().get(dv, 0) for dv in dtau_set)
            
            if dtau_degree <= 1:
                result += term
        return result
    
    return expr.replace(
        lambda f: isinstance(f, AppliedUndef),
        lambda f: f.func(clean_arg(f.args[0]))
    )

In [10]:
temp = expand(Dn(eta[(1,2)],(1,2),(3,1),(1,3)))
display(Math(latex(temp)))

display(Math(latex(drop_dtau_squared_in_args(temp))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
X2 = expand(
      P12_X2(eta[(1,2)]) + P21_X2(eta[(2,1)])
    + P13_X2(eta[(1,3)]) + P31_X2(eta[(3,1)])
)

In [12]:
display(Math(r'X_2 = ' + color_terms(latex(X2))))

<IPython.core.display.Math object>

In [13]:
RT12 = r[(1,2)] + Dn(r[(2,1)], (1,2))   # r12 + D_12 * r21
RT13 = r[(1,3)] + Dn(r[(3,1)], (1,3))   # r13 + D_13 * r31

R_X2 = {}

# (41a): R12 = -(1 - D131)(r12 + D12*r21) + (2 - D121 - D12131)(r13 + D13*r31)
R_X2[(1,2)] = (
    - (RT12 - Dn(RT12, (3,1),(1,3)))
    + (2*RT13 - Dn(RT13, (2,1),(1,2)) - Dn(RT13, (3,1),(1,3),(2,1),(1,2)))
)

# (41b): R23 = 0
R_X2[(2,3)] = 0

# (41c): R31 = -(2 - D131 - D13121)(r12 + D12*r21)
#             + (1 - D121)(r13 + D13*r31)
#             + (1 - D121 - D12131 - D1312121)*r13
R_X2[(3,1)] = (
    - (2*RT12 - Dn(RT12, (3,1),(1,3)) - Dn(RT12, (2,1),(1,2),(3,1),(1,3)))
    + (RT13 - Dn(RT13, (2,1),(1,2)))
    + (r[(1,3)]
       - Dn(r[(1,3)], (2,1),(1,2))
       - Dn(r[(1,3)], (3,1),(1,3),(2,1),(1,2))
       + Dn(r[(1,3)], (2,1),(1,2),(2,1),(1,2),(3,1),(1,3)))  # <- PLUS not minus
)

R_X2[(2,1)] = (
    - (RT12 - Dn(RT12, (3,1),(1,3)))
    - (r[(1,2)]
       - Dn(r[(1,2)], (3,1),(1,3))
       - Dn(r[(1,2)], (2,1),(1,2),(3,1),(1,3))
       + Dn(r[(1,2)], (3,1),(1,3),(3,1),(1,3),(2,1),(1,2)))  # <- PLUS not minus
    + (2*RT13 - Dn(RT13, (2,1),(1,2)) - Dn(RT13, (3,1),(1,3),(2,1),(1,2)))
)

# (41e): R32 = 0
R_X2[(3,2)] = 0

# (41f): R13 = -(2 - D131 - D13121)(r12 + D12*r21)
#             + (1 - D121)(r13 + D13*r31)
R_X2[(1,3)] = (
    - (2*RT12 - Dn(RT12, (3,1),(1,3)) - Dn(RT12, (2,1),(1,2),(3,1),(1,3)))
    + (RT13 - Dn(RT13, (2,1),(1,2)))
)

In [14]:
a = {(i,j): sc[i][2] - sc[j][2] for (i,j) in tau}

correction_X2 = 0
for (i,j,k) in [(1,2,3), (2,3,1), (3,1,2)]:
    correction_X2 -= (-a[(i,j)]*R_X2[(i,j)] - a[(i,k)]*R_X2[(i,k)])


X2c = expand(X2 + correction_X2)

In [15]:
#display(Math(r'X_2 = ' + color_terms(latex(X2))))

In [16]:
display(Math(r'X_2 = ' + color_terms(latex(X2c))))

<IPython.core.display.Math object>

In [17]:
REF_AB= REF[(1,3)] - REF[(2,3)]
REF_AC= REF[(3,2)] - REF[(1,2)]
REF_CB= REF[(2,1)] - REF[(3,1)]

display(Math(r'REF_{AB} = ' + color_terms(latex(REF_AB))))
display(Math(r'REF_{AC} = ' + color_terms(latex(REF_AC))))
display(Math(r'REF_{CB} = ' + color_terms(latex(REF_CB))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [18]:
Corr1 = omega1*(-Dn(REF_AB, (1, 2)) + Dn(REF_AB, (2,1), (1,2)) + Dn(REF_AB, (1,2),(3,1), (1,3)) - Dn(REF_AB, (2,1), (1,2), (3,1), (1,3))
                +Dn(REF_AB, (1,2), (2,1), (1,2), (3,1), (1,3)) - Dn(REF_AB, (2,1), (1,2), (2,1), (1,2), (3,1), (1,3)) - Dn(REF_AB, (1,2), (3,1), (1,3), (3,1), (1,3), (2,1), (1,2)))

Corr2 = omega1*(-Dn(REF_AC, (1, 3)) + Dn(REF_AC, (3,1), (1,3)) + Dn(REF_AC, (1,3),(2,1), (1,2)) - Dn(REF_AC, (3,1), (1,3), (2,1), (1,2))
                +Dn(REF_AC, (1,3), (3,1), (1,3), (2,1), (1,2)) - Dn(REF_AC, (3,1), (1,3), (3,1), (1,3), (2,1), (1,2)) - Dn(REF_AC, (1,3), (2,1), (1,2), (2,1), (1,2), (3,1), (1,3))) 



Corr3 = omega1*(Dn(-REF_CB, (3,1), (1,3), (2,1), (1,2), (2,1), (1,2), (3,1), (1,3)))  # Both should have the same effect
Corr3 = omega1*(Dn(-REF_CB, (2,1), (1,2), (3,1), (1,3), (3,1), (1,3), (2,1), (1,2))) 


X2cc = expand(X2c + Corr1 + Corr2 + Corr3)
display(Math(r'X_2 = ' + color_terms(latex(X2cc))))

<IPython.core.display.Math object>

In [19]:
display(Math(r'X_2 = ' + color_terms(latex(drop_dtau_squared_in_args(X2cc)))))

<IPython.core.display.Math object>